# AMEX Feature Ablation And OOF Diagnostics

Single Colab notebook for the extra diagnostics requested after the public AMEX portfolio reconstruction.

What this notebook can do if the original Drive artifacts exist:

- find the original AMEX artifact folder used by `Untitled41.ipynb`,
- load full OOF prediction files,
- compute model correlation,
- compute equal-blend and leave-one-model-out AMEX metrics,
- compute incremental gain vs the best single model and all-model blend,
- optionally run feature-view ablation CV for basic summary, change/ratio, temporal, recent-window, and pivot-lite feature sets.

This notebook does not fabricate missing results. If required OOF or feature parquet files are absent, it prints what is missing.

In [ ]:
# Colab setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or failed:', exc)

import os
import gc
import glob
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
except Exception as exc:
    raise RuntimeError('scikit-learn is required for this notebook') from exc

try:
    import lightgbm as lgb
except Exception:
    print('Installing lightgbm...')
    !pip -q install lightgbm
    import lightgbm as lgb

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

# OOF diagnostics are fast if OOF files exist.
RUN_OOF_DIAGNOSTICS = True

# Feature ablation retrains models and can take a long time. Toggle when ready.
RUN_FEATURE_ABLATION = False

# Use a smaller subset only for a pipeline smoke test. Do not publish smoke-test metrics.
SMOKE_TEST_ROWS = None  # e.g. 50000 for debugging only

RANDOM_STATE = 42
N_SPLITS = 5

## 1. Locate Original AMEX Artifact Folder

`Untitled41.ipynb` mainly used `MyDrive/amex_features`. This cell checks the same locations and creates a public output folder for new aggregate diagnostics.

In [ ]:
CANDIDATE_BASE_DIRS = [
    Path('/content/drive/MyDrive/amex_features'),
    Path('/content/gdrive/MyDrive/amex_features'),
]

BASE_DIR = next((p for p in CANDIDATE_BASE_DIRS if p.exists()), None)
if BASE_DIR is None:
    print('No AMEX artifact folder found. Checked:')
    for p in CANDIDATE_BASE_DIRS:
        print(' -', p)
else:
    print('Using BASE_DIR:', BASE_DIR)

OUT_DIR = (BASE_DIR / 'portfolio_extra_diagnostics' if BASE_DIR else Path('/content/portfolio_extra_diagnostics'))
TABLE_DIR = OUT_DIR / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print('Output table dir:', TABLE_DIR)

if BASE_DIR:
    inventory = []
    for path in sorted(BASE_DIR.glob('*')):
        if path.is_file():
            inventory.append({'file_name': path.name, 'size_mb': path.stat().st_size / 1024 / 1024})
    inventory_df = pd.DataFrame(inventory).sort_values('size_mb', ascending=False)
    display(inventory_df.head(80))

## 2. Metric Functions

AMEX metric implementation matching the competition structure: weighted top-four-percent capture plus normalized weighted Gini.

In [ ]:
def _validate_binary_metric_input(y_true, y_score):
    labels = np.asarray(y_true).astype(int)
    scores = np.asarray(y_score).astype(float)
    if len(labels) != len(scores):
        raise ValueError('y_true and y_score length mismatch')
    if set(np.unique(labels)) != {0, 1}:
        raise ValueError('AMEX metric requires both classes')
    return labels, scores


def top_four_percent_captured(y_true, y_score):
    labels, scores = _validate_binary_metric_input(y_true, y_score)
    df = pd.DataFrame({'target': labels, 'prediction': scores})
    df = df.sort_values('prediction', ascending=False, kind='mergesort').reset_index(drop=True)
    df['weight'] = np.where(df['target'].eq(0), 20, 1)
    cutoff = max(int(0.04 * df['weight'].sum()), 1)
    selected = df.loc[df['weight'].cumsum().le(cutoff)]
    return selected.loc[selected['target'].eq(1), 'weight'].sum() / df.loc[df['target'].eq(1), 'weight'].sum()


def weighted_gini(y_true, y_score):
    labels, scores = _validate_binary_metric_input(y_true, y_score)
    df = pd.DataFrame({'target': labels, 'prediction': scores})
    df['weight'] = np.where(df['target'].eq(0), 20, 1)
    df = df.sort_values('prediction', ascending=False, kind='mergesort').reset_index(drop=True)
    random = (df['weight'] / df['weight'].sum()).cumsum()
    total_pos = (df['target'] * df['weight']).sum()
    lorentz = (df['target'] * df['weight']).cumsum() / total_pos
    return ((lorentz - random) * df['weight']).sum()


def normalized_gini(y_true, y_score):
    return weighted_gini(y_true, y_score) / weighted_gini(y_true, y_true)


def amex_metric(y_true, y_score):
    return 0.5 * (normalized_gini(y_true, y_score) + top_four_percent_captured(y_true, y_score))


def score_binary_ranking(y_true, y_score):
    return {
        'roc_auc': roc_auc_score(y_true, y_score),
        'amex_metric': amex_metric(y_true, y_score),
    }

## 3. OOF Diagnostics

This section needs full OOF files. If they exist in Drive, it produces:

- `oof_model_correlation.csv`,
- `oof_single_and_leave_one_out_blend.csv`,
- `oof_incremental_gain.csv`.

In [ ]:
OOF_FILE_CANDIDATES = {
    'lgb_full': ['oof_full_lgbm.csv'],
    'xgb_full': ['oof_full_xgb_gpu.csv'],
    'lgb_top1600_dart': ['oof_lgbm_top1600_dart_5fold.csv'],
    'cat_full': ['oof_catboost_full_5fold.csv', 'oof_cat_full.csv'],
    'lgb_top1600_goss': ['oof_lgbm_top1600_goss_5fold.csv'],
    'lgb_recent_change_goss': ['oof_lgbm_recent_change_goss_5fold.csv'],
    'mlp_full': ['oof_mlp_residual_gelu_bnln_onecycle_5fold.csv'],
    'pivot_lite_lgbm': ['oof_pivot_lite_lgbm_5fold.csv', 'oof_train_pivot_lite_lgbm_5fold.csv'],
}


def find_first_existing(base_dir, candidates):
    for name in candidates:
        path = base_dir / name
        if path.exists():
            return path
    return None


def detect_prediction_column(df, target_col='target'):
    excluded = {'customer_ID', target_col, 'fold'}
    candidates = [c for c in df.columns if c not in excluded]
    numeric = [c for c in candidates if pd.api.types.is_numeric_dtype(df[c])]
    if len(numeric) != 1:
        raise ValueError(f'Expected exactly one numeric prediction column, found {numeric}')
    return numeric[0]


def load_oof_matrix(base_dir, target_col='target'):
    found = {}
    missing = {}
    for model_name, candidates in OOF_FILE_CANDIDATES.items():
        path = find_first_existing(base_dir, candidates)
        if path is None:
            missing[model_name] = candidates
        else:
            found[model_name] = path

    print('Found OOF files:')
    for model_name, path in found.items():
        print(f' - {model_name}: {path.name}')
    if missing:
        print('\nMissing OOF files:')
        for model_name, candidates in missing.items():
            print(f' - {model_name}: one of {candidates}')

    merged = None
    for model_name, path in found.items():
        df = pd.read_csv(path)
        if 'customer_ID' not in df.columns or target_col not in df.columns:
            raise ValueError(f'{path.name} must include customer_ID and {target_col}')
        pred_col = detect_prediction_column(df, target_col=target_col)
        df = df[['customer_ID', target_col, pred_col]].rename(columns={pred_col: model_name})
        merged = df if merged is None else merged.merge(df, on=['customer_ID', target_col], how='inner')

    if merged is None:
        return None, found, missing
    return merged, found, missing


def run_oof_diagnostics(oof_df, prediction_cols, target_col='target'):
    corr = oof_df[prediction_cols].corr(method='pearson')

    rows = []
    for col in prediction_cols:
        scores = score_binary_ranking(oof_df[target_col], oof_df[col])
        rows.append({'evaluation_type': 'single_model', 'model_name': col, 'removed_model': '', 'n_models': 1, **scores})

    oof_df = oof_df.copy()
    oof_df['all_model_equal_blend'] = oof_df[prediction_cols].mean(axis=1)
    all_scores = score_binary_ranking(oof_df[target_col], oof_df['all_model_equal_blend'])
    rows.append({'evaluation_type': 'all_model_equal_blend', 'model_name': ' + '.join(prediction_cols), 'removed_model': '', 'n_models': len(prediction_cols), **all_scores})

    for removed in prediction_cols:
        kept = [c for c in prediction_cols if c != removed]
        blend_col = f'without_{removed}'
        oof_df[blend_col] = oof_df[kept].mean(axis=1)
        scores = score_binary_ranking(oof_df[target_col], oof_df[blend_col])
        rows.append({'evaluation_type': 'leave_one_out_blend', 'model_name': ' + '.join(kept), 'removed_model': removed, 'n_models': len(kept), **scores})

    comparison = pd.DataFrame(rows)
    all_metric = comparison.loc[comparison['evaluation_type'].eq('all_model_equal_blend'), 'amex_metric'].iloc[0]
    best_single = comparison.loc[comparison['evaluation_type'].eq('single_model'), 'amex_metric'].max()
    single = comparison[comparison['evaluation_type'].eq('single_model')][['model_name', 'roc_auc', 'amex_metric']].copy()
    single = single.rename(columns={'roc_auc': 'single_model_roc_auc', 'amex_metric': 'single_model_amex'})
    loo = comparison[comparison['evaluation_type'].eq('leave_one_out_blend')][['removed_model', 'amex_metric']].copy()
    loo['leave_one_out_delta_vs_all'] = all_metric - loo['amex_metric']
    loo = loo.rename(columns={'removed_model': 'model_name', 'amex_metric': 'blend_without_model_amex'})
    gain = single.merge(loo, on='model_name', how='left')
    gain['all_model_blend_amex'] = all_metric
    gain['best_single_amex'] = best_single
    gain['best_single_delta_vs_model'] = gain['single_model_amex'] - best_single
    gain['all_blend_delta_vs_model'] = all_metric - gain['single_model_amex']
    gain = gain.sort_values('leave_one_out_delta_vs_all', ascending=False).reset_index(drop=True)
    return corr, comparison, gain

In [ ]:
if RUN_OOF_DIAGNOSTICS and BASE_DIR:
    oof_df, found_oof, missing_oof = load_oof_matrix(BASE_DIR)
    if oof_df is None:
        print('No OOF diagnostics were run because no OOF files were found.')
    else:
        prediction_cols = list(found_oof.keys())
        print('Merged OOF shape:', oof_df.shape)
        print('Prediction columns:', prediction_cols)
        display(oof_df.head())

        corr, blend_comparison, incremental_gain = run_oof_diagnostics(oof_df, prediction_cols)
        corr.to_csv(TABLE_DIR / 'oof_model_correlation.csv')
        blend_comparison.to_csv(TABLE_DIR / 'oof_single_and_leave_one_out_blend.csv', index=False)
        incremental_gain.to_csv(TABLE_DIR / 'oof_incremental_gain.csv', index=False)

        print('Saved OOF diagnostics to:', TABLE_DIR)
        display(corr)
        display(blend_comparison.sort_values(['evaluation_type', 'amex_metric'], ascending=[True, False]))
        display(incremental_gain)
else:
    print('OOF diagnostics skipped. Set RUN_OOF_DIAGNOSTICS=True and make sure BASE_DIR exists.')

## 4. Feature-View Ablation CV

This section answers: what happens with only basic customer-level summary variables, then with added change/ratio, temporal, recent-window, and pivot-lite representations?

It retrains LightGBM for each feature view. Keep the same folds and model parameters across views. This can be slow.

In [ ]:
TRAIN_FEATURE_PATH = BASE_DIR / 'train_feat_no_diffmono_v1.parquet' if BASE_DIR else None
PIVOT_LITE_PATH = BASE_DIR / 'train_feat_pivot_lite_v1.parquet' if BASE_DIR else None

LGB_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 64,
    'max_depth': -1,
    'min_child_samples': 100,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_estimators': 5000,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbose': -1,
}

FEATURE_VIEW_SPECS = [
    {
        'feature_view': 'basic_summary_only',
        'input_kind': 'aggregate',
        'include_any': ['_last', '_first', '_mean', '_std', '_min', '_max', '_sum', '_median', '_count', '_nunique'],
        'exclude_any': ['lag', 'recent', 'minus', 'div', 'missing'],
    },
    {
        'feature_view': 'summary_plus_change_ratio',
        'input_kind': 'aggregate',
        'include_any': ['_last', '_first', '_mean', '_std', '_min', '_max', '_sum', '_median', '_count', '_nunique', '_minus_', '_div_'],
        'exclude_any': ['lag', 'recent'],
    },
    {
        'feature_view': 'summary_change_temporal',
        'input_kind': 'aggregate',
        'include_any': ['_last', '_first', '_mean', '_std', '_min', '_max', '_sum', '_median', '_count', '_nunique', '_minus_', '_div_', 'lag'],
        'exclude_any': ['recent'],
    },
    {
        'feature_view': 'summary_change_temporal_recent',
        'input_kind': 'aggregate',
        'include_any': ['_last', '_first', '_mean', '_std', '_min', '_max', '_sum', '_median', '_count', '_nunique', '_minus_', '_div_', 'lag', 'recent'],
        'exclude_any': [],
    },
    {
        'feature_view': 'pivot_lite',
        'input_kind': 'pivot_lite',
        'include_any': [''],
        'exclude_any': [],
    },
]


def select_feature_columns(columns, include_any, exclude_any):
    protected = {'customer_ID', 'target'}
    date_like = [c for c in columns if 'date' in c.lower()]
    protected.update(date_like)
    selected = []
    for col in columns:
        if col in protected:
            continue
        include = any(pattern in col for pattern in include_any)
        exclude = any(pattern in col for pattern in exclude_any)
        if include and not exclude:
            selected.append(col)
    return selected


def clean_feature_frame(df, feature_cols):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    numeric_cols = X.select_dtypes(include='number').columns.tolist()
    X = X[numeric_cols]
    nunique = X.nunique(dropna=False)
    non_constant = nunique[nunique > 1].index.tolist()
    X = X[non_constant]
    return X.fillna(X.median(numeric_only=True))


def train_lgbm_cv_for_view(df, feature_cols, run_name):
    y = df['target'].astype(int).reset_index(drop=True)
    X = clean_feature_frame(df.reset_index(drop=True), feature_cols)
    if SMOKE_TEST_ROWS is not None:
        X = X.iloc[:SMOKE_TEST_ROWS].reset_index(drop=True)
        y = y.iloc[:SMOKE_TEST_ROWS].reset_index(drop=True)
        print('SMOKE TEST MODE: metrics are not publishable')

    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X), dtype=float)
    fold_rows = []

    for fold, (train_idx, valid_idx) in enumerate(folds.split(X, y), start=1):
        model = lgb.LGBMClassifier(**LGB_PARAMS)
        callbacks = [lgb.early_stopping(200), lgb.log_evaluation(100)]
        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            eval_set=[(X.iloc[valid_idx], y.iloc[valid_idx])],
            callbacks=callbacks,
        )
        pred = model.predict_proba(X.iloc[valid_idx])[:, 1]
        oof[valid_idx] = pred
        fold_score = score_binary_ranking(y.iloc[valid_idx], pred)
        fold_rows.append({'feature_view': run_name, 'fold': fold, 'n_features': X.shape[1], 'best_iteration': model.best_iteration_, **fold_score})
        print(run_name, 'fold', fold, fold_score)
        gc.collect()

    overall = score_binary_ranking(y, oof)
    oof_out = pd.DataFrame({'target': y, f'oof_{run_name}': oof})
    if 'customer_ID' in df.columns:
        oof_out.insert(0, 'customer_ID', df['customer_ID'].iloc[:len(oof_out)].values)
    return overall, pd.DataFrame(fold_rows), oof_out, X.shape[1]


def path_for_view(spec):
    return PIVOT_LITE_PATH if spec['input_kind'] == 'pivot_lite' else TRAIN_FEATURE_PATH

In [ ]:
if RUN_FEATURE_ABLATION and BASE_DIR:
    rows = []
    fold_tables = []
    for spec in FEATURE_VIEW_SPECS:
        input_path = path_for_view(spec)
        if input_path is None or not input_path.exists():
            print('Missing feature file for', spec['feature_view'], ':', input_path)
            continue

        print('\n' + '=' * 80)
        print('Feature view:', spec['feature_view'])
        print('Input:', input_path)
        train_df = pd.read_parquet(input_path)
        if 'target' not in train_df.columns:
            raise ValueError(f'{input_path.name} must contain target for CV ablation')
        feature_cols = select_feature_columns(list(train_df.columns), spec['include_any'], spec['exclude_any'])
        print('Selected raw feature count:', len(feature_cols))
        if not feature_cols:
            print('No features selected; skipping')
            continue

        overall, fold_df, oof_out, clean_n_features = train_lgbm_cv_for_view(train_df, feature_cols, spec['feature_view'])
        rows.append({
            'feature_view': spec['feature_view'],
            'input_file': input_path.name,
            'selected_feature_count': len(feature_cols),
            'clean_numeric_feature_count': clean_n_features,
            'validation_type': '5-fold OOF',
            **overall,
        })
        fold_tables.append(fold_df)
        oof_path = TABLE_DIR / f"oof_feature_view_{spec['feature_view']}.csv"
        oof_out.to_csv(oof_path, index=False)
        print('Saved OOF:', oof_path)
        del train_df, oof_out
        gc.collect()

    ablation = pd.DataFrame(rows).sort_values('amex_metric', ascending=False) if rows else pd.DataFrame()
    if not ablation.empty:
        ablation.to_csv(TABLE_DIR / 'feature_view_ablation_results.csv', index=False)
        pd.concat(fold_tables, ignore_index=True).to_csv(TABLE_DIR / 'feature_view_ablation_fold_results.csv', index=False)
        print('Saved feature ablation results to:', TABLE_DIR)
        display(ablation)
else:
    print('Feature ablation skipped. Set RUN_FEATURE_ABLATION=True when ready to retrain CV models.')
    if TRAIN_FEATURE_PATH:
        print('Aggregate feature path exists:', TRAIN_FEATURE_PATH.exists(), TRAIN_FEATURE_PATH)
    if PIVOT_LITE_PATH:
        print('Pivot-lite feature path exists:', PIVOT_LITE_PATH.exists(), PIVOT_LITE_PATH)

## 5. README-Ready Summary Text

This cell prints a conservative write-up only from computed outputs. If no diagnostics were computed, it says so.

In [ ]:
summary_lines = []

corr_path = TABLE_DIR / 'oof_model_correlation.csv'
gain_path = TABLE_DIR / 'oof_incremental_gain.csv'
ablation_path = TABLE_DIR / 'feature_view_ablation_results.csv'

if gain_path.exists():
    gain = pd.read_csv(gain_path)
    all_blend = gain['all_model_blend_amex'].iloc[0]
    best_single = gain['best_single_amex'].iloc[0]
    top_contributor = gain.sort_values('leave_one_out_delta_vs_all', ascending=False).iloc[0]
    summary_lines.append(f"The all-model equal blend recorded AMEX Metric {all_blend:.6f}, compared with the best single model AMEX Metric {best_single:.6f}.")
    summary_lines.append(f"In leave-one-out diagnostics, removing {top_contributor['model_name']} changed the blend AMEX Metric by {top_contributor['leave_one_out_delta_vs_all']:.6f}.")
else:
    summary_lines.append('OOF contribution diagnostics were not computed because full OOF files were not available or RUN_OOF_DIAGNOSTICS was disabled.')

if ablation_path.exists():
    ablation = pd.read_csv(ablation_path).sort_values('amex_metric', ascending=False)
    best = ablation.iloc[0]
    basic = ablation[ablation['feature_view'].eq('basic_summary_only')]
    if not basic.empty:
        summary_lines.append(f"Using only basic customer-level summary features recorded AMEX Metric {basic.iloc[0]['amex_metric']:.6f}.")
    summary_lines.append(f"The best feature-view ablation was {best['feature_view']} with AMEX Metric {best['amex_metric']:.6f}.")
else:
    summary_lines.append('Feature-view ablation metrics were not computed because feature parquet files were unavailable or RUN_FEATURE_ABLATION was disabled.')

print('\n'.join(summary_lines))
(TABLE_DIR / 'readme_ready_extra_diagnostics_summary.txt').write_text('\n'.join(summary_lines), encoding='utf-8')